In [10]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import classification_report

# 1. HARDWARE & PATHS
# Forcing the environment to only see one GPU ensures no "device mismatch" errors
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")

# Set paths
LORA_PATH = "lora_model"  # Path to your fine-tuned adapters
TEST_DATA_PATH = "data/test.out"

# 2. CONFIGURATION
max_seq_length = 2048
dtype = torch.bfloat16 # Use None for auto-detection or torch.float16 for older GPUs
load_in_4bit = False 

# 3. LOAD MODEL & TOKENIZER
# If you have a fine-tuned model, load_name should be LORA_PATH
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = LORA_PATH, 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Unsloth logic to move to GPU and optimize
FastLanguageModel.for_inference(model)

# 4. LABEL DEFINITIONS
ALL_LABELS = ["ADMINISTERED_TO", "AFFECTS", "ASSOCIATED_WITH", "AUGMENTS", "CAUSES", 
              "COEXISTS_WITH", "COMPARED_WITH", "DIAGNOSES", "DISRUPTS", "INHIBITS", 
              "INTERACTS_WITH", "ISA", "LOCATION_OF", "None", "PART_OF", "PRECEDES", 
              "PREDISPOSES", "PREVENTS", "PROCESS_OF", "PRODUCES", "STIMULATES", "TREATS", "USES"]

# 5. DATA PREPARATION
def format_prompt(row):
    labels_str = ", ".join(ALL_LABELS)
    messages = [
        {
            "role": "system", 
            "content": f"You are a biomedical expert. Classify the relationship. Allowed labels: {labels_str}. Output ONLY the label name."
        },
        {
            "role": "user", 
            "content": f"Sentence: {row['sentence']}\nSubject: {row['subject_text']} ({row['subject_type']})\nObject: {row['object_text']} ({row['object_type']})\nLabel:"
        }
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def map_response_to_label(generated_text, label_list):
    gen_clean = generated_text.strip().upper()
    for l in label_list:
        if gen_clean == l.upper():
            return l
    for l in label_list:
        if l != "None" and l.lower() in generated_text.lower():
            return l
    return "None"

def load_test_data(path):
    df = pd.read_csv(path, sep="\t", header=None, keep_default_na=False, nrows=100)
    df.columns = ["idx", "label", "sentence", "subject_text", "object_text", 
                  "subject_type", "object_type", "subject_group", "object_group", "label_choices"]
    return df

# 6. INFERENCE ENGINE
def run_test_and_save(df, output_csv="final_test_results.csv"):
    predictions = []
    raw_responses = []
    
    print(f"Running inference on {len(df)} samples on GPU:{GPU_ID}...")
    
    # Set model to evaluation mode
    model.eval()
    
    with torch.no_grad():
        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt_text = format_prompt(row)
            
            # Explicitly send inputs to the same device as the model
            inputs = tokenizer([prompt_text], return_tensors="pt").to(device)
            
            outputs = model.generate(
                **inputs, 
                max_new_tokens=10, 
                use_cache=True,
                temperature=0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            
            # Move output back to CPU for decoding to free up GPU memory logic
            generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
            raw_responses.append(generated_text)
            
            final_pred = map_response_to_label(generated_text, ALL_LABELS)
            predictions.append(final_pred)

    df['model_raw_response'] = raw_responses
    df['predicted_label'] = predictions
    df.to_csv(output_csv, index=False, sep="\t")
    return predictions



==((====))==  Unsloth 2025.5.3: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA H100 NVL. Num GPUs = 4. Max memory: 93.086 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
test_df = load_test_data(TEST_DATA_PATH)
print(TEST_DATA_PATH)

data/test.out


In [12]:
# 7. EXECUTION
# preds = run_test_and_save(test_df)
# print("\n" + "="*30)
# print("TEST SET EVALUATION")
# print("="*30)
# print(classification_report(test_df['label'], preds, zero_division=0))

In [13]:
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    prompt_text = format_prompt(row)
    print(prompt_text)
    break
    inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")
        

  0%|          | 0/100 [00:00<?, ?it/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a biomedical expert. Classify the relationship. Allowed labels: ADMINISTERED_TO, AFFECTS, ASSOCIATED_WITH, AUGMENTS, CAUSES, COEXISTS_WITH, COMPARED_WITH, DIAGNOSES, DISRUPTS, INHIBITS, INTERACTS_WITH, ISA, LOCATION_OF, None, PART_OF, PRECEDES, PREDISPOSES, PREVENTS, PROCESS_OF, PRODUCES, STIMULATES, TREATS, USES. Output ONLY the label name.<|eot_id|><|start_header_id|>user<|end_header_id|>

Sentence: Here we show that chymases, which are chymotryptic peptidases secreted by mast cells, hydrolyze HGF, thereby abolishing scatter factor activity while generating an NK4-like antagonist of HGF scatter factor activity.
Subject: chymases (Amino Acid, Peptide, or Protein)
Object: scatter factor (Amino Acid, Peptide, or Protein)
Label:<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [ ]:
preds = run_test_and_save(test_df)
print(test_df['label'])

Running inference on 100 samples on GPU:0...


100%|██████████| 100/100 [00:14<00:00,  7.10it/s]

['INHIBITS', 'None', 'ISA', 'AUGMENTS', 'INTERACTS_WITH', 'PREVENTS', 'INTERACTS_WITH', 'PREVENTS', 'None', 'INTERACTS_WITH', 'AUGMENTS', 'CAUSES', 'ADMINISTERED_TO', 'None', 'None', 'None', 'COEXISTS_WITH', 'None', 'None', 'None', 'COEXISTS_WITH', 'None', 'COEXISTS_WITH', 'None', 'ADMINISTERED_TO', 'None', 'None', 'None', 'ISA', 'COEXISTS_WITH', 'None', 'None', 'DIAGNOSES', 'PRODUCES', 'None', 'None', 'None', 'AUGMENTS', 'None', 'AUGMENTS', 'None', 'None', 'None', 'None', 'INTERACTS_WITH', 'COEXISTS_WITH', 'None', 'ADMINISTERED_TO', 'None', 'None', 'None', 'ADMINISTERED_TO', 'COEXISTS_WITH', 'None', 'None', 'INHIBITS', 'DIAGNOSES', 'None', 'None', 'ADMINISTERED_TO', 'None', 'None', 'AFFECTS', 'COEXISTS_WITH', 'None', 'DIAGNOSES', 'None', 'AFFECTS', 'AFFECTS', 'COEXISTS_WITH', 'None', 'None', 'ISA', 'COEXISTS_WITH', 'None', 'None', 'None', 'None', 'ISA', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'COEXISTS_WITH', 'AUGMENTS', 'None', 'COEXISTS_WITH', 'None', 'INTERACTS_WITH

In [18]:
print(test_df['label'].to_list())

['INHIBITS', 'PART_OF', 'ISA', 'INHIBITS', 'None', 'PREVENTS', 'None', 'PREVENTS', 'None', 'None', 'INHIBITS', 'CAUSES', 'USES', 'None', 'None', 'PART_OF', 'None', 'PART_OF', 'None', 'None', 'None', 'None', 'None', 'None', 'COMPARED_WITH', 'None', 'None', 'None', 'ISA', 'LOCATION_OF', 'None', 'None', 'CAUSES', 'LOCATION_OF', 'PROCESS_OF', 'None', 'ISA', 'AUGMENTS', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'TREATS', 'LOCATION_OF', 'None', 'None', 'None', 'None', 'ISA', 'None', 'None', 'None', 'None', 'TREATS', 'ADMINISTERED_TO', 'None', 'None', 'None', 'None', 'None', 'DIAGNOSES', 'None', 'LOCATION_OF', 'None', 'None', 'AFFECTS', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'ISA', 'None', 'None', 'PART_OF', 'None', 'None', 'LOCATION_OF', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'None', 'LOCATION_OF']


In [ ]:
print(classification_report(test_df['label'], preds, zero_division=0))

                 precision    recall  f1-score   support

ADMINISTERED_TO       0.20      1.00      0.33         1
        AFFECTS       0.00      0.00      0.00         1
       AUGMENTS       0.20      1.00      0.33         1
         CAUSES       1.00      0.50      0.67         2
  COEXISTS_WITH       0.00      0.00      0.00         0
  COMPARED_WITH       0.00      0.00      0.00         1
      DIAGNOSES       0.33      1.00      0.50         1
       INHIBITS       0.50      0.33      0.40         3
 INTERACTS_WITH       0.00      0.00      0.00         0
            ISA       0.75      0.60      0.67         5
    LOCATION_OF       0.00      0.00      0.00         6
           None       0.79      0.65      0.71        69
        PART_OF       0.00      0.00      0.00         4
       PREVENTS       1.00      1.00      1.00         2
     PROCESS_OF       0.00      0.00      0.00         1
       PRODUCES       0.00      0.00      0.00         0
         TREATS       0.00    

In [39]:
df = pd.read_csv("data/train.out", sep="\t", header=None, keep_default_na=False)
df.columns = ["idx", "label", "sentence", "subject_text", "object_text", 
                  "subject_type", "object_type", "subject_group", "object_group", "label_choices"]
target_labels = ALL_LABELS[:5]
few_shot = []
for label  in ALL_LABELS[:5]:
    a = df[df['label']== label]
    few_shot.append(a['idx'].iloc[0])
print(few_shot)

[np.int64(88), np.int64(7), np.int64(5), np.int64(9), np.int64(34)]


In [44]:
print(ALL_LABELS[:5])

['ADMINISTERED_TO', 'AFFECTS', 'ASSOCIATED_WITH', 'AUGMENTS', 'CAUSES']


In [40]:
few_shot = df[df['idx'].isin(few_shot)]
few_shot.reset_index(drop=True, inplace=True)
df_train = df[~df['idx'].isin(few_shot)]
df_train.reset_index(drop=True, inplace=True)

In [43]:
few_shot.to_csv("data/few_shot.out", sep="\t", header=False, index=False)
df_train.to_csv("data/new_train.out", sep="\t", header=False, index=False)

In [42]:
df_train

,idx,label,sentence,subject_text,object_text,subject_type,object_type,subject_group,object_group,label_choices
0,0,LOCATION_OF,We tested the hsp-70 promoter as well as a pro...,head,reporter gene,Body Location or Region,Gene or Genome,Anatomy,Genes and Molecular Sequences,location_of
1,1,PROCESS_OF,A patient who had had a functioning peritoneov...,superior vena cava syndrome,patient,Disease or Syndrome,Human,Disorders,Living Beings,process_of; affects
2,2,None,Antiestrogenic effect of clomiphene in the hum...,Antiestrogenic,clomiphene,Pharmacologic Substance,Organic Chemical,Chemicals and Drugs,Chemicals and Drugs,stimulates; coexists_with; inhibits; interacts...
3,3,None,These results demonstrate that [3H]BTX-B can b...,sodium channel,antiarrhythmic drugs,"Amino Acid, Peptide, or Protein",Pharmacologic Substance,Chemicals and Drugs,Chemicals and Drugs,inhibits; compared_with; interacts_with; stimu...
4,4,PART_OF,The main site for ROS production is the respir...,respiratory chain,mitochondria,Cell Component,Cell Component,Anatomy,Anatomy,part_of; location_of
...,...,...,...,...,...,...,...,...,...,...
8182,8182,INHIBITS,"Furthermore, neferine significantly decreased ...",neferine,IL-6,Organic Chemical,"Amino Acid, Peptide, or Protein",Chemicals and Drugs,Chemicals and Drugs,stimulates; coexists_with; inhibits; interacts...
8183,8183,None,Tyrosine peptides as precursors of melanin in ...,Tyrosine,mammals,Pharmacologic Substance,Mammal,Chemicals and Drugs,Living Beings,treats; administered_to
8184,8184,AFFECTS,These data point to a specific role for IL-7/I...,STAT5,transcriptional,Gene or Genome,Genetic Function,Genes and Molecular Sequences,Physiology,location_of; augments; affects
8185,8185,INTERACTS_WITH,Effect of isoniazid on preservation of nicotin...,dinucleotide,isoniazid,"Nucleic Acid, Nucleoside, or Nucleotide",Organic Chemical,Chemicals and Drugs,Chemicals and Drugs,interacts_with
